# K4TQ-F3 "Freddy 3": as fast as a 15-inch FPV quad with its payload

Freddy's airframe (1.2 m, front NACA 4415 / rear NACA 2415) with the Jetson Orin NX and the perception (as A2), 1 kg payload; the drive swept for the top speed — **MN505-S KV380 + 16x12 on the same 6S3P**.

**The target**: the 15-inch FPV quad's top speed carrying a payload (`k4tq_quads.fpv15`, the same catalogue and drive
model: 15x8 on MN5006 KV450, 6S2P Li-ion) — ≈34.5 m/s (124 km/h); K4TQ-F3 must match or exceed it with four motors
running, with a 5 % margin (the model's 1 m/s grid and its uncertainty).

**The rules** (`k4tq_fast.sweep` / `pick`): a motor bigger than Freddy autonomous (F2)'s; T/W ≥ 1.6 on the sagged pack, most loaded motor
≤ 75 % in the hover; R3 closes (10 km radius, 2 min hover, 20 % reserve, 8 m/s wind) with ≥ 20 Wh to spare; the root's
actuator (sized for the top speed + 10 %), bearings and locking pin within their ratings; the lightest that passes (then the most R3 margin).

Why the drive and not the airframe: Freddy autonomous's top speed is its propeller's pitch speed, not its power —
the drag at 35 m/s is only ≈0.5 kW of thrust power. A higher pitch (or a higher pack voltage) moves it.
Labels: ESTIMATE / CALC (MOCK BEMT under `DEBUG = True`); the new parts are "to verify".

In [ ]:
DEBUG = True
from vegeta.labkit import bootstrap
DEBUG = bootstrap.lab("aero", DEBUG)

In [ ]:
import pandas as pd
from IPython.display import Image, display
import k4tq, k4tq_sizing as sz, k4tq_fast as kf, k4tq_quads as kq
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 50)
LETTER, PARENT = "F3", "F2"
print(k4tq.params(LETTER)["title"])
print("target: the 15-inch FPV quad's top speed with payload", kf.target_speed(), "m/s")

## 1. The sweep (motor × propeller × pack) — from `data/k4tq/sweep_F3.csv`; re-run below (10–30 min)

In [ ]:
sw = pd.read_csv(sz.DATA / "sweep_F3.csv")
sw.attrs["target speed [m/s]"] = kf.target_speed()
# sw = kf.sweep(LETTER); sw.to_csv(sz.DATA / "sweep_F3.csv", index=False)       # the full sweep again
cols = ["motor", "prop", "pack", "top speed [m/s]", "take-off mass [kg]", "payload [kg]", "T/W sagged", "R3 margin [Wh]", "still-air range [km]",
        "hover [W]", "stall [m/s]", "actuator reserve", "meets the target", "note"]
sw[[c for c in cols if c in sw]].sort_values("top speed [m/s]", ascending=False).round(2)

In [ ]:
print("the pick:"); kf.pick(LETTER, sw)

## 2. Against Freddy autonomous (F2) and the 15-inch FPV quad

In [ ]:
df = kf.comparison(designs=(PARENT, LETTER))
df.T

## 3. Power and range against speed

In [ ]:
Image(str(kf.figure(designs=(PARENT, LETTER))))

## 4. The root and the hover

In [ ]:
v = sz.vehicle(LETTER)
r = sz.root_loads(v)
display(pd.Series({k: r[k] for k in ("case governing the bending", "root moment ultimate [N·m]", "actuator design torque [N·m]", "actuator available at the spar [N·m]",
                    "actuator reserve", "bearing static reserve", "locking pin shear stress, double shear [MPa]", "spar chosen")}))
pd.Series(sz.hover(v))

## Reading it

- **Top speed**: ≈42 m/s (151 km/h) against the FPV's 124 km/h with 1 kg — 22 % faster, with a 1 kg payload, the Jetson
  and 53–56 km of still-air range (the FPV: ≈20 km with 1 kg).
- **What it took**: the MN505-S KV380 (+55 g per motor) and the 16x12 propeller on the same 6S3P pack: +0.25 kg over
  Freddy autonomous. The higher pitch costs a little in the hover (≈1.33 kW against 1.18 kW) and in the R3 margin
  (21 Wh against 24 Wh), and lifts the T/W to 1.85 (F2 was 1.60, just under the rule).
- **The structure**: the actuator now sized for 46 m/s (top + 10 %): reserve 2.1. The wing's divergence and flutter at
  42 m/s are a Stage 4 check (the all-moving wing on its drive).
- **The FPV still wins** the hover (24 min with 1 kg against 13) and the mass (3.3 kg against 7.5).